# Week 2 — Context Engineering I

**AI Agentic Engineering · Corte 1**

Companion notebook to `week-02-context-engineering-i-content.html`.

**You will practice:**
1. System prompt vs. no system prompt, same question.
2. Zero-shot vs. few-shot prompting on a classification task.
3. Chain-of-thought vs. direct answer on a multi-step problem.
4. Schema-forced JSON output — raw SDK, then a light ADK and LangChain preview.
5. Two open exercises.


In [ ]:
%pip install -q --upgrade google-genai google-adk langchain-google-genai langgraph python-dotenv pydantic

In [ ]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel

load_dotenv()
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
MODEL = "gemini-flash-latest"

## 1. System prompt vs. no system prompt

Same user question, with and without a system instruction.

In [1]:
import ollama

MODEL = "qwen2.5:3b"
question = "My bike's front brake feels loose, what should I check?"

# --- Sin system prompt (Respuesta streaming) ---
print("--- Without system prompt ---")
stream_no_system = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": question}],
    stream=True  # Imprime token por token
)

for chunk in stream_no_system:
    print(chunk['message']['content'], end='', flush=True)

print("\n\n" + "="*50 + "\n")

# --- Con system prompt (Respuesta streaming) ---
print("--- With system prompt ---")
stream_with_system = ollama.chat(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": (
                "You are a support assistant for a bike rental company. "
                "Only answer questions related to bike rentals, maintenance, and safety. "
                "Keep answers under 3 sentences. If asked something unrelated, politely decline."
            )
        },
        {"role": "user", "content": question}
    ],
    stream=True
)

for chunk in stream_with_system:
    print(chunk['message']['content'], end='', flush=True)

--- Without system prompt ---
If your bike's front brake feels loose, you should check the following components and adjustments:

1. **Brake Calipers**: Ensure that the brake calipers are properly aligned and not rubbing against the rim. Calipers should slide smoothly but not rub against the wheel during braking.

2. **Brake Levers**: Make sure that the brake levers are adjusted properly. The levers should be straight and not bent. They should pull the calipers inward to apply the brakes correctly.

3. **Brake Pads**: Check the condition of the brake pads. They should be evenly worn and not excessively worn or excessively worn on one side. If they are worn, consider replacing them.

4. **Brake Holes and Adjusters**: Ensure that the brake holes and adjusters are correctly adjusted. The calipers should be able to move freely towards and away from the wheel hub.

5. **Brake Hose and Cable**: Check for any kinks or damage in the brake hose or cable. Ensure that the brake hose is tight and 

## 2. Zero-shot vs. few-shot

Watch the output format stabilize once examples are added.

In [2]:
import ollama

MODEL = "qwen2.5:14b"

zero_shot = 'Classify the sentiment of this review as positive, negative, or neutral: "It\'s fine, does what it says on the box."'

few_shot = '''Classify the sentiment of a review as positive, negative, or neutral.

Review: "Fast shipping and the product works great."
Sentiment: positive

Review: "It arrived broken and support never replied."
Sentiment: negative

Review: "It's fine, does what it says on the box."
Sentiment:'''

# Ejecución Zero-shot
zero_shot_res = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": zero_shot}]
)

# Ejecución Few-shot
few_shot_res = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": few_shot}]
)

print("Zero-shot:", zero_shot_res['message']['content'].strip())
print("Few-shot: ", few_shot_res['message']['content'].strip())

Zero-shot: The sentiment of the review "It's fine, does what it says on the box" can be classified as neutral. The phrase indicates that the product meets basic expectations without any notable positive or negative attributes.
Few-shot:  neutral


## 3. Chain-of-thought vs. direct answer

In [3]:
import ollama

MODEL = "qwen2.5:14b"
problem = "A store had 120 apples. It sold 35% of them in the morning and 28 more in the afternoon. How many apples are left?"

# --- Direct Prompting ---
direct = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "user", "content": problem + " Answer with just the number."}
    ],
    options={"temperature": 0.1}
)
print("Direct:", direct['message']['content'].strip())

# --- Chain-of-Thought (CoT) Prompting ---
cot = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "user", "content": problem + ' Think step by step, then give the final answer on its own line starting with "Answer:".'}
    ],
    options={"temperature": 0.1}
)
print("\nChain-of-thought:\n", cot['message']['content'].strip())

Direct: 44

Chain-of-thought:
 First, calculate how many apples were sold in the morning. Since 35% of the apples were sold, we calculate 35% of 120:

\[ 35\% \times 120 = 0.35 \times 120 = 42 \]

So, 42 apples were sold in the morning.

Next, we know that 28 more apples were sold in the afternoon.

Now, let's find out how many apples were sold in total:

\[ 42 + 28 = 70 \]

The store originally had 120 apples. After selling 70 apples, the number of apples left is:

\[ 120 - 70 = 50 \]

Answer: 50


## 4. Schema-forced JSON output

### 4a. Raw SDK

In [5]:
from typing import Literal
from pydantic import BaseModel
import ollama

MODEL = "qwen2.5:14b"


# Usamos Literal para restringir los valores permitidos en el esquema JSON
class TicketTriage(BaseModel):
  category: Literal["billing", "technical", "account", "other"]
  urgency: Literal["low", "medium", "high"]
  summary: str


response = ollama.chat(
    model=MODEL,
    messages=[{
        "role": "user",
        "content": (
            "My card was charged twice for the same order and I need this"
            " fixed today."
        ),
    }],
    format=TicketTriage.model_json_schema(),
    options={"temperature": 0.1},
)

ticket = TicketTriage.model_validate_json(response["message"]["content"])
print(ticket)
print(ticket.category, "|", ticket.urgency)

category='billing' urgency='high' summary='Double charge on card for same order needs resolution today.'
billing | high


### 4b. Google ADK — `output_schema` (light preview)

ADK's `LlmAgent` accepts a Pydantic model directly via `output_schema` and returns validated structured output.

In [6]:
import ollama


def ask_adk_agent(prompt: str) -> str:
  response = ollama.chat(
      model=MODEL,
      messages=[
          {
              "role": "system",
              "content": "Triage the support message into the given schema.",
          },
          {"role": "user", "content": prompt},
      ],
      format=TicketTriage.model_json_schema(),
      options={"temperature": 0.0},
  )
  return response["message"]["content"]


raw_json = ask_adk_agent(
    "My card was charged twice for the same order and I need this fixed today."
)
print(raw_json)
print(TicketTriage.model_validate_json(raw_json))

{
  "category": "billing",
  "urgency": "high",
  "summary": "Duplicate charge for a single order transaction that needs immediate resolution."
}
category='billing' urgency='high' summary='Duplicate charge for a single order transaction that needs immediate resolution.'


### 4c. LangChain — `with_structured_output` (light preview)

In [7]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model=MODEL, temperature=0.0)
structured_llm = llm.with_structured_output(TicketTriage)

result = structured_llm.invoke(
    "My card was charged twice for the same order and I need this fixed today."
)
print(result)

category='billing' urgency='high' summary='Customer needs a double charge on their card for the same order to be resolved immediately.'


## 5. Exercises

In [8]:
import ollama

MODEL = "qwen2.5:14b"

# 1. Definición de las 3 entradas de prueba
test_inputs = [
    "I need to update my credit card before my subscription expires tomorrow.",
    "The export button in the reports tab isn't responding when clicked.",
    "Where can I find the invoice for my purchase last month?",
]

# 2. Prompts: Zero-Shot vs Few-Shot
zero_shot_template = """Classify the following message into one of these exact categories: [Urgent-Billing, Standard-Billing, Low-Technical, High-Technical, Standard-Account].
Output ONLY the category name.

Message: {input_text}
Category:"""

few_shot_template = """Classify the following message into one of these exact categories: [Urgent-Billing, Standard-Billing, Low-Technical, High-Technical, Standard-Account].
Output ONLY the category name.

Message: "Our entire database connection is failing and all users are locked out."
Category: High-Technical

Message: "I was charged $50 instead of $20 and my account is suspended!"
Category: Urgent-Billing

Message: "How do I change my profile picture?"
Category: Standard-Account

Message: "The app crashed once when I opened settings."
Category: Low-Technical

Message: {input_text}
Category:"""


# 3. Función de prueba
def run_classification(prompt_text):
  response = ollama.chat(
      model=MODEL,
      messages=[{"role": "user", "content": prompt_text}],
      options={"temperature": 0.0},
  )
  return response["message"]["content"].strip()


# 4. Ejecución y comparación en los 3 test inputs
print("=== COMPARISON RESULTS ===")
for i, text in enumerate(test_inputs, 1):
  zs_prompt = zero_shot_template.format(input_text=text)
  fs_prompt = few_shot_template.format(input_text=text)

  zs_output = run_classification(zs_prompt)
  fs_output = run_classification(fs_prompt)

  print(f"\n--- Test Input {i} ---")
  print(f"Input: '{text}'")
  print(f"Zero-Shot Output: {zs_output}")
  print(f"Few-Shot Output : {fs_output}")

=== COMPARISON RESULTS ===

--- Test Input 1 ---
Input: 'I need to update my credit card before my subscription expires tomorrow.'
Zero-Shot Output: Urgent-Billing
Few-Shot Output : Urgent-Billing

--- Test Input 2 ---
Input: 'The export button in the reports tab isn't responding when clicked.'
Zero-Shot Output: High-Technical
Few-Shot Output : Low-Technical

--- Test Input 3 ---
Input: 'Where can I find the invoice for my purchase last month?'
Zero-Shot Output: Urgent-Billing
Few-Shot Output : Standard-Billing


In [9]:
from pydantic import BaseModel, Field
import ollama

MODEL = "qwen2.5:14b"


# 1. Definición del modelo Pydantic
class ResumeLine(BaseModel):
  role: str = Field(
      description="The job title or role, e.g., 'Senior backend engineer'"
  )
  company: str = Field(description="The company name, e.g., 'Northwind Traders'")
  years: float = Field(
      description="Duration in years as a float, e.g., 3.5"
  )


# 2. Entrada de texto libre
input_text = "Senior backend engineer at Northwind Traders for 3.5 years"

# 3. Llamada con formato forzado por esquema JSON
response = ollama.chat(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": (
                "Extract job details from the input text strictly matching the"
                " provided JSON schema."
            ),
        },
        {"role": "user", "content": input_text},
    ],
    format=ResumeLine.model_json_schema(),
    options={"temperature": 0.0},
)

# 4. Parseo y validación del objeto
parsed_resume = ResumeLine.model_validate_json(response["message"]["content"])

print("Parsed Object:")
print(parsed_resume)
print("\nIndividual Fields:")
print("Role   :", parsed_resume.role)
print("Company:", parsed_resume.company)
print("Years  :", parsed_resume.years)

Parsed Object:
role='Senior backend engineer' company='Northwind Traders' years=3.5

Individual Fields:
Role   : Senior backend engineer
Company: Northwind Traders
Years  : 3.5


## Next week

Week 3 — **Context Engineering II**: managing the context window (summarization, compression, sliding window)
and an introduction to embeddings and semantic search. See `week-03-context-engineering-ii-content.html`.